In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/mrajareivan/data-understanding-aml/__results__.html
/kaggle/input/notebooks/mrajareivan/data-understanding-aml/AML_data_from_eda.parquet
/kaggle/input/notebooks/mrajareivan/data-understanding-aml/__notebook__.ipynb
/kaggle/input/notebooks/mrajareivan/data-understanding-aml/__output__.json
/kaggle/input/notebooks/mrajareivan/data-understanding-aml/custom.css
/kaggle/input/notebooks/mrajareivan/data-understanding-aml/__results___files/__results___47_0.png
/kaggle/input/notebooks/mrajareivan/data-understanding-aml/__results___files/__results___31_0.png
/kaggle/input/notebooks/mrajareivan/data-understanding-aml/__results___files/__results___53_0.png
/kaggle/input/notebooks/mrajareivan/data-understanding-aml/__results___files/__results___35_0.png
/kaggle/input/notebooks/mrajareivan/data-understanding-aml/__results___files/__results___56_0.png
/kaggle/input/notebooks/mrajareivan/data-understanding-aml/__results___files/__results___27_0.png
/kaggle/input/notebooks/mraj

In [3]:
df = pd.read_parquet("/kaggle/input/notebooks/mrajareivan/data-understanding-aml/AML_data_from_eda.parquet")
df.head()

,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type,Datetime,is_cross_border,is_currency_conversion,hour,day_of_week,date_only,day_of_month
0,1900-01-01 10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits,2022-10-07 10:35:19,0,0,10,4,2022-10-07,7
1,1900-01-01 10:35:20,2022-10-07,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border,0,Normal_Fan_Out,2022-10-07 10:35:20,1,1,10,4,2022-10-07,7
2,1900-01-01 10:35:20,2022-10-07,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,Cheque,0,Normal_Small_Fan_Out,2022-10-07 10:35:20,0,0,10,4,2022-10-07,7
3,1900-01-01 10:35:21,2022-10-07,5376652437,9600420220,11895.00,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Fan_In,2022-10-07 10:35:21,0,0,10,4,2022-10-07,7
4,1900-01-01 10:35:21,2022-10-07,9614186178,3803336972,115.25,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits,2022-10-07 10:35:21,0,0,10,4,2022-10-07,7


In [4]:
df.columns

Index(['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount',
       'Payment_currency', 'Received_currency', 'Sender_bank_location',
       'Receiver_bank_location', 'Payment_type', 'Is_laundering',
       'Laundering_type', 'Datetime', 'is_cross_border',
       'is_currency_conversion', 'hour', 'day_of_week', 'date_only',
       'day_of_month'],
      dtype='object')

In [5]:
df['Datetime'].is_monotonic_increasing

True

In [6]:
## kurang minute dan months
df["minute"] = df["Datetime"].dt.minute
df["month"] = df["Datetime"].dt.minute

In [7]:
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [8]:
## Melakukan Cyclical Encoding (rekayasa fitur siklik)
## Mengubah data waktu linear (jam, hari, bulan) menjadi koordinat lingkaran 2D (sin & cos).
## Gunanya agar model memahami sifat waktu yang berulang (misal: jam 23 dan jam 0 itu berdampingan, atau Desember dan Januari itu berurutan).
df = df.assign(
    hour_sin  = np.sin(2 * np.pi * df["hour"] / 24),
    hour_cos  = np.cos(2 * np.pi * df["hour"] / 24),
    dow_sin   = np.sin(2 * np.pi * df["day_of_week"] / 7),
    dow_cos   = np.cos(2 * np.pi * df["day_of_week"] / 7),
    month_sin = np.sin(2 * np.pi * df["month"] / 12),
    month_cos = np.cos(2 * np.pi * df["month"] / 12)
)


In [9]:
df["log_amount"] = np.log1p(df["Amount"])

In [10]:
df["account_pair"] = (df["Sender_account"].astype(str) + "_" + df["Receiver_account"].astype(str))

In [11]:
df["sender_tx_count_history"] = (df.groupby("Sender_account").cumcount())

In [12]:
sender_cumsum = (
    df.groupby("Sender_account")["Amount"]
         .cumsum()
)

df["sender_total_amount_history"] = (
    sender_cumsum - df["Amount"]
)

In [13]:
df[df['Sender_account']==1491989064][["Amount", "log_amount", "sender_tx_count_history", "sender_total_amount_history"]].head()

,Amount,log_amount,sender_tx_count_history,sender_total_amount_history
1,6019.64,8.702949,0,0.00
196,12782.41,9.455904,1,6019.64
204,26553.78,10.186965,2,18802.05
445,6037.94,8.705984,3,45355.83
1083,8284.53,9.022266,4,51393.77


In [14]:
df["sender_avg_amount_history"] = np.where(
    df["sender_tx_count_history"] > 0,
    df["sender_total_amount_history"]
    / df["sender_tx_count_history"],
    0
)

df["sender_amount_vs_history_ratio"] = (
    df["Amount"]
    / (df["sender_avg_amount_history"] + 1e-6)
)

In [15]:
new_receiver = (
    ~df.duplicated(
        subset=["Sender_account", "Receiver_account"],
        keep="first"
    )
)

df["sender_new_receiver"] = new_receiver.astype(int)

df["sender_unique_receivers_history"] = (
    df.groupby("Sender_account")["sender_new_receiver"]
         .cumsum()
         - df["sender_new_receiver"]
)

In [16]:
df["receiver_tx_count_history"] = (
    df.groupby("Receiver_account")
         .cumcount()
)

In [17]:
receiver_cumsum = (
    df.groupby("Receiver_account")["Amount"]
         .cumsum()
)

df["receiver_total_amount_history"] = (
    receiver_cumsum - df["Amount"]
)

In [ ]:
df["receiver_avg_amount_history"] = np.where(
    df["receiver_tx_count_history"] > 0,
    df["receiver_total_amount_history"]
    / df["receiver_tx_count_history"],
    0
)

In [19]:
df["receiver_amount_vs_history_ratio"] = (
    df["Amount"]
    / (df["receiver_avg_amount_history"] + 1e-6)
)

In [20]:
new_sender = (
    ~df.duplicated(
        subset=["Receiver_account", "Sender_account"],
        keep="first"
    )
)

df["receiver_new_sender"] = new_sender.astype(int)

df["receiver_unique_senders_history"] = (
    df.groupby("Receiver_account")["receiver_new_sender"]
         .cumsum()
         - df["receiver_new_sender"]
)

In [21]:
df["sender_time_since_last_tx"] = (
    df.groupby("Sender_account")["Datetime"]
         .diff()
         .dt.total_seconds()
)

df["receiver_time_since_last_tx"] = (
    df.groupby("Receiver_account")["Datetime"]
         .diff()
         .dt.total_seconds()
)



In [22]:
import numpy as np
import pandas as pd


def add_rolling_features(
    df,
    account_col,
    prefix,
    windows=("1h", "24h")
):
    """
    Add causal rolling transaction count and amount.

    Only transactions strictly before the current transaction
    are included.

    Memory-efficient:
    - No merge()
    - No temporary row-id column
    - One rolling operation per window
    - Keeps only required columns in temporary dataframe
    """

    # --------------------------------------------------
    # 1. Simpan posisi asli dataframe
    # --------------------------------------------------
    original_index = df.index

    # --------------------------------------------------
    # 2. Ambil hanya kolom yang diperlukan
    # --------------------------------------------------
    temp = df[[account_col, "Datetime", "Amount"]].copy()

    # --------------------------------------------------
    # 3. Sort berdasarkan account + waktu
    # --------------------------------------------------
    temp = temp.sort_values(
        [account_col, "Datetime"],
        kind="mergesort"
    )

    # Index setelah sorting = posisi baris asli
    sorted_index = temp.index

    # --------------------------------------------------
    # 4. Set Datetime sebagai time index
    # --------------------------------------------------
    temp = temp.set_index("Datetime")

    # --------------------------------------------------
    # 5. Group berdasarkan account
    # --------------------------------------------------
    grouped = temp.groupby(account_col)["Amount"]

    # --------------------------------------------------
    # 6. Hitung setiap rolling window
    # --------------------------------------------------
    for window in windows:

        rolling = (
            grouped
            .rolling(
                window=window,
                closed="left"
            )
            .agg(["count", "sum"])
        )

        # rolling memiliki MultiIndex:
        # account_col + Datetime
        #
        # Urutannya sama dengan temp setelah sorting.

        count_values = rolling["count"].to_numpy()
        amount_values = rolling["sum"].to_numpy()

        # --------------------------------------------------
        # 7. Kembalikan hasil ke urutan dataframe asli
        # --------------------------------------------------
        count_series = pd.Series(
            count_values,
            index=sorted_index
        )

        amount_series = pd.Series(
            amount_values,
            index=sorted_index
        )

        # Reindex mengikuti dataframe original
        df[f"{prefix}_tx_count_{window}"] = (
            count_series.reindex(original_index).to_numpy()
        )

        df[f"{prefix}_total_amount_{window}"] = (
            amount_series.reindex(original_index).to_numpy()
        )

        del rolling
        del count_values
        del amount_values
        del count_series
        del amount_series

    del temp
    del grouped

    return df

In [23]:
df = add_rolling_features(
    df,
    account_col="Sender_account",
    prefix="sender"
)

In [24]:
df = add_rolling_features(
    df,
    account_col="Receiver_account",
    prefix="receiver"
)

In [25]:
## FOR SENDER
df["sender_avg_amount_1h"] = (
    df["sender_total_amount_1h"]
    / (df["sender_tx_count_1h"] + 1e-6)
)

df["sender_avg_amount_24h"] = (
    df["sender_total_amount_24h"]
    / (df["sender_tx_count_24h"] + 1e-6)
)

## FOR RECEIVER
df["receiver_avg_amount_1h"] = (
    df["receiver_total_amount_1h"]
    / (df["receiver_tx_count_1h"] + 1e-6)
)

df["receiver_avg_amount_24h"] = (
    df["receiver_total_amount_24h"]
    / (df["receiver_tx_count_24h"] + 1e-6)
)

In [26]:
df["sender_amount_vs_1h_avg"] = (
    df["Amount"]
    / (df["sender_avg_amount_1h"] + 1e-6)
)
df["sender_amount_vs_24h_avg"] = (
    df["Amount"]
    / (df["sender_avg_amount_24h"] + 1e-6)
)

df["receiver_amount_vs_1h_avg"] = (
    df["Amount"]
    / (df["receiver_avg_amount_1h"] + 1e-6)
)
df["receiver_amount_vs_24h_avg"] = (
    df["Amount"]
    / (df["receiver_avg_amount_24h"] + 1e-6)
)

In [27]:
df = df.drop(
    columns=[
        "sender_account_age_hours",
        "receiver_account_age_hours"
    ],
    errors="ignore"
)

In [28]:
df["sender_is_new_account"] = (
    df.groupby("Sender_account")
         .cumcount()
    == 0
).astype(int)

df["receiver_is_new_account"] = (
    df.groupby("Receiver_account")
         .cumcount()
    == 0
).astype(int)

In [29]:
df["pair_tx_count_history"] = (
    df.groupby(
        ["Sender_account", "Receiver_account"]
    ).cumcount()
)

df["is_repeated_pair"] = (
    df["pair_tx_count_history"] > 0
).astype(int)

In [30]:
rolling_cols = [
    col
    for col in df.columns
    if any(
        keyword in col
        for keyword in [
            "_tx_count_1h",
            "_tx_count_24h",
            "_total_amount_1h",
            "_total_amount_24h"
        ]
    )
]

df[rolling_cols] = df[rolling_cols].fillna(0)

In [ ]:
avg_cols = [
    "sender_avg_amount_history",
    "receiver_avg_amount_history",
    "sender_avg_amount_1h",
    "sender_avg_amount_24h",
    "receiver_avg_amount_1h",
    "receiver_avg_amount_24h",
]

ratio_cols = [
    "sender_amount_vs_history_ratio",
    "sender_amount_vs_1h_avg",
    "sender_amount_vs_24h_avg",
    "receiver_amount_vs_history_ratio",
    "receiver_amount_vs_1h_avg",
    "receiver_amount_vs_24h_avg",
]

df[avg_cols] = df[avg_cols].fillna(0)
df[ratio_cols] = df[ratio_cols].fillna(0)

In [31]:
df["sender_time_since_last_tx"] = (
    df["sender_time_since_last_tx"]
    .fillna(-1)
)

df["receiver_time_since_last_tx"] = (
    df["receiver_time_since_last_tx"]
    .fillna(-1)
)

In [32]:
feature_cols = [
    # Transaction
    "Amount",
    "log_amount",

    # Temporal
    "hour",
    "minute",
    "day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",

    # Transaction characteristics
    "is_cross_border",
    "is_currency_conversion",

    # Sender history
    "sender_tx_count_history",
    "sender_total_amount_history",
    "sender_avg_amount_history",
    "sender_amount_vs_history_ratio",
    "sender_unique_receivers_history",
    "sender_time_since_last_tx",
    "sender_tx_count_1h",
    "sender_total_amount_1h",
    "sender_avg_amount_1h",
    "sender_tx_count_24h",
    "sender_total_amount_24h",
    "sender_avg_amount_24h",
    "sender_amount_vs_1h_avg",
    "sender_amount_vs_24h_avg",

    # Receiver history
    "receiver_tx_count_history",
    "receiver_total_amount_history",
    "receiver_avg_amount_history",
    "receiver_amount_vs_history_ratio",
    "receiver_unique_senders_history",
    "receiver_time_since_last_tx",
    "receiver_tx_count_1h",
    "receiver_total_amount_1h",
    "receiver_avg_amount_1h",
    "receiver_tx_count_24h",
    "receiver_total_amount_24h",
    "receiver_avg_amount_24h",
    "receiver_amount_vs_1h_avg",
    "receiver_amount_vs_24h_avg",

    # Pair behavior
    "pair_tx_count_history",
    "is_repeated_pair",

    # New account indicators
    "sender_is_new_account",
    "receiver_is_new_account",
]

target_col = "Is_laundering"

In [41]:
X = df[feature_cols].copy()
y = df[target_col].copy()

In [42]:
print(X.shape)
display(X.head())

(9504852, 44)


,Amount,log_amount,hour,minute,day_of_week,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,...,receiver_avg_amount_1h,receiver_tx_count_24h,receiver_total_amount_24h,receiver_avg_amount_24h,receiver_amount_vs_1h_avg,receiver_amount_vs_24h_avg,pair_tx_count_history,is_repeated_pair,sender_is_new_account,receiver_is_new_account
0,1459.15,7.286294,10,35,4,0,0.5,-0.866025,-0.433884,-0.900969,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,1
1,6019.64,8.702949,10,35,4,0,0.5,-0.866025,-0.433884,-0.900969,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,1
2,14328.44,9.570071,10,35,4,0,0.5,-0.866025,-0.433884,-0.900969,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,1
3,11895.00,9.383957,10,35,4,0,0.5,-0.866025,-0.433884,-0.900969,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,1
4,115.25,4.755743,10,35,4,0,0.5,-0.866025,-0.433884,-0.900969,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,1


In [43]:
# Missing
print("Missing values:")
display(X.isna().sum()[X.isna().sum() > 0])

# Infinite
print("Infinite values:")
print(np.isinf(X.select_dtypes(include=np.number)).sum())

# Target distribution
print("\nTarget:")
print(y.value_counts())

Missing values:


Series([], dtype: int64)

Infinite values:
Amount                              0
log_amount                          0
hour                                0
minute                              0
day_of_week                         0
is_weekend                          0
hour_sin                            0
hour_cos                            0
dow_sin                             0
dow_cos                             0
is_cross_border                     0
is_currency_conversion              0
sender_tx_count_history             0
sender_total_amount_history         0
sender_avg_amount_history           0
sender_amount_vs_history_ratio      0
sender_unique_receivers_history     0
sender_time_since_last_tx           0
sender_tx_count_1h                  0
sender_total_amount_1h              0
sender_avg_amount_1h                0
sender_tx_count_24h                 0
sender_total_amount_24h             0
sender_avg_amount_24h               0
sender_amount_vs_1h_avg             0
sender_amount_vs_24h_avg         

In [44]:
X.describe().T

,count,mean,std,min,25%,50%,75%,max
Amount,9504852.0,8.762968e+03,2.561495e+04,3.730000,2143.687500,6113.720000,1.045846e+04,1.261850e+07
log_amount,9504852.0,8.347972e+00,1.402560e+00,1.553925,7.670749,8.718454,9.255262e+00,1.635067e+01
hour,9504852.0,1.418820e+01,5.714591e+00,0.000000,10.000000,14.000000,1.900000e+01,2.300000e+01
minute,9504852.0,2.950190e+01,1.731770e+01,0.000000,14.000000,30.000000,4.400000e+01,5.900000e+01
day_of_week,9504852.0,2.972619e+00,1.998519e+00,0.000000,1.000000,3.000000,5.000000e+00,6.000000e+00
is_weekend,9504852.0,2.798203e-01,4.489108e-01,0.000000,0.000000,0.000000,1.000000e+00,1.000000e+00
hour_sin,9504852.0,-2.201548e-01,6.574704e-01,-1.000000,-0.866025,-0.500000,2.588190e-01,1.000000e+00
hour_cos,9504852.0,-1.853863e-01,6.963451e-01,-1.000000,-0.866025,-0.500000,5.000000e-01,1.000000e+00
dow_sin,9504852.0,5.951520e-03,7.051679e-01,-0.974928,-0.781831,0.000000,7.818315e-01,9.749279e-01
dow_cos,9504852.0,1.615809e-03,7.090136e-01,-0.900969,-0.900969,-0.222521,6.234898e-01,1.000000e+00


In [47]:
sample_account = df["Sender_account"].iloc[0]

display(
    df[
        df["Sender_account"] == sample_account
    ][[
        "Datetime",
        "Sender_account",
        "Receiver_account",
        "Amount",
        "sender_tx_count_history",
        "sender_unique_receivers_history",
        "sender_tx_count_1h",
        "sender_tx_count_24h"
    ]].head(20)
)

,Datetime,Sender_account,Receiver_account,Amount,sender_tx_count_history,sender_unique_receivers_history,sender_tx_count_1h,sender_tx_count_24h
0,2022-10-07 10:35:19,8724731955,2769355426,1459.15,0,0,0.0,0.0
1179863,2022-11-16 15:13:25,8724731955,2769355426,4699.43,1,1,0.0,0.0
2642489,2023-01-05 16:40:54,8724731955,2769355426,3175.15,2,1,0.0,0.0
3085200,2023-01-20 17:57:47,8724731955,2769355426,4879.60,3,1,0.0,0.0
3214446,2023-01-25 07:54:11,8724731955,2769355426,798.96,4,1,0.0,0.0
4037748,2023-02-20 10:29:41,8724731955,2769355426,335.67,5,1,0.0,0.0
4709517,2023-03-14 10:46:59,8724731955,2769355426,341.48,6,1,0.0,0.0
4807910,2023-03-17 15:37:23,8724731955,2769355426,1090.97,7,1,0.0,0.0
5306414,2023-04-03 15:02:51,8724731955,2769355426,713.51,8,1,0.0,0.0
6754162,2023-05-22 10:14:20,8724731955,2769355426,295.97,9,1,0.0,0.0


In [48]:
df.to_parquet('AML_data_feature_engineering.parquet')